# AgriShield: Agricultural Price Crash Prediction & Early Warning System
### End-to-End Machine Learning Model Training, Benchmarking & Evaluation Pipeline

**Author**: AgriShield Engineering & Data Science Team  
**Objective**: Predict multi-day crop prices (Model A - Regression) and assess the risk of a severe price crash (Model B - Classification) across Indian mandis.

---

## 1. Environment Setup & Dataset Loading

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set project root
ROOT_DIR = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()
sys.path.insert(0, str(ROOT_DIR))

from ml.config import DATASET_PATH, DEFAULT_CRASH_THRESHOLD_PCT, DEFAULT_FORECAST_DAYS
from ml.data_preprocessing import load_raw_dataset, validate_and_clean_data, chronological_split
from ml.feature_engineering import create_time_series_features, generate_targets, prepare_ml_matrices, FEATURE_COLUMNS

df = pd.read_csv(DATASET_PATH)
print(f"Loaded raw dataset: {df.shape[0]} rows, {df.shape[1]} columns.")
df.head()

## 2. Data Validation & Preprocessing

In [2]:
clean_df, report = validate_and_clean_data(df)
print("Validation Report:")
for k, v in report.items():
    print(f"  {k}: {v}")

## 3. Exploratory Data Analysis (EDA)

In [3]:
# Price Distribution by Major Crops
plt.figure(figsize=(12, 5))
sns.boxplot(x="crop", y="modal_price", data=clean_df)
plt.xticks(rotation=45)
plt.title("Modal Price Distribution by Crop (₹/Quintal)")
plt.ylabel("Price (₹)")
plt.grid(True, alpha=0.3)
plt.show()

## 4. Feature Engineering & Target Generation (Zero Lookahead Leakage)

In [4]:
# Demonstrating time-series lag and rolling statistics generation
sample_crop_df = clean_df[(clean_df['crop'] == 'Tomato') & (clean_df['mandi'] == 'Oddanchatram')].copy()
sample_feat_df = create_time_series_features(sample_crop_df)
print(f"Engineered {len(FEATURE_COLUMNS)} features without future lookahead bias.")
sample_feat_df[['date', 'modal_price', 'price_lag_1', 'price_lag_7', 'rolling_mean_7', 'arrivals_surge_ratio']].tail()

## 5. Strict Chronological Train/Validation/Test Split

In [5]:
train_raw, test_raw = chronological_split(clean_df, test_days=60)
print(f"Train period: {train_raw['date'].min().date()} to {train_raw['date'].max().date()} ({len(train_raw)} records)")
print(f"Test period:  {test_raw['date'].min().date()} to {test_raw['date'].max().date()} ({len(test_raw)} records)")

X_train, Y_train_reg, Y_train_clf, _ = prepare_ml_matrices(train_raw, is_training=True)
X_test, Y_test_reg, Y_test_clf, _ = prepare_ml_matrices(test_raw, is_training=True)
print(f"Train samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

## 6. Baseline Models vs Advanced Ensembles (Model A - Regression)

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 1. Baseline
lr_base = MultiOutputRegressor(LinearRegression())
lr_base.fit(X_train_scaled, Y_train_reg)
preds_lr = lr_base.predict(X_test_scaled)
print(f"Linear Regression Baseline MAE: ₹{mean_absolute_error(Y_test_reg, preds_lr):.2f}")

# 2. Random Forest Regressor
rf_reg = RandomForestRegressor(n_estimators=80, max_depth=12, random_state=42, n_jobs=-1)
rf_reg.fit(X_train_scaled, Y_train_reg)
preds_rf = rf_reg.predict(X_test_scaled)
print(f"Random Forest Regressor MAE:    ₹{mean_absolute_error(Y_test_reg, preds_rf):.2f}")

## 7. Model B: Price Crash Risk (Classification) & Class Imbalance Handling

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# Baseline Logistic Regression
lr_clf = LogisticRegression(class_weight='balanced', max_iter=400)
lr_clf.fit(X_train_scaled, Y_train_clf)
preds_lr_clf = lr_clf.predict(X_test_scaled)
print("Logistic Regression Baseline F1:", f1_score(Y_test_clf, preds_lr_clf, zero_division=0))

# Random Forest Classifier with balanced subsample
rf_clf = RandomForestClassifier(n_estimators=90, max_depth=10, class_weight='balanced_subsample', random_state=42)
rf_clf.fit(X_train_scaled, Y_train_clf)
preds_rf_clf = rf_clf.predict(X_test_scaled)
print("Random Forest Classifier F1:", f1_score(Y_test_clf, preds_rf_clf, zero_division=0))
print(classification_report(Y_test_clf, preds_rf_clf))

## 8. Model Saving, Inference Example & Limitations

In [8]:
from ml.predict import get_predictor
predictor = get_predictor()
prediction = predictor.predict_crop_mandi("Tomato", "Tamil Nadu", "Oddanchatram", forecast_days=7)
print("Inference Result:")
print(f"Crop: {prediction['crop']} at {prediction['mandi']}")
print(f"Current Price: ₹{prediction['current_price']}/qtl")
print(f"Crash Probability: {prediction['crash_probability_pct']}%")
print(f"Risk Level: {prediction['risk_category']}")
print(f"Tomorrow's Forecast: ₹{prediction['tomorrow_predicted_price']}/qtl")

### Conclusion and Operational Limitations
1. **Statistical Nature**: Predictions are statistical expectations based on past arrivals and seasonal trends; sudden geopolitical shocks or natural disasters cannot be forecasted.
2. **Data Coverage**: Fresh, daily mandi reporting is required for maximum accuracy.
3. **Farmer Guidance**: Farmers should combine these predictions with local intelligence before committing harvests.